> ### 💾 Saving Your Work — Read This First
>
> **Villanova does not use Google Workspace**, so Colab will NOT auto-save unless you sign in with a personal Google account.
>
> **Option A — Save to GitHub (recommended):** File → Save a copy in GitHub → choose your course repo
>
> **Option B — Download:** File → Download → Download .ipynb
>
> **Save at the end of every section.**


# Lab 13: Can We Predict How Well a Radio Station Comes In?
## Building a Signal Strength Model with RadioLand Data

**CSC 2053 — Spring 2026**

---

You've spent the last few weeks analyzing radio station data with Pandas and NumPy.
Today you're going to use it to *predict* something real: **how strong a radio signal
is at a specific receiving location.**

The RadioLand app predicts field strength in **dBu** (decibels above one microvolt per meter).
Higher = clearer reception. Lower = static.

RadioLand uses the Longley-Rice time-tested propagation model to make its predictions.

Your goal: build a machine learning model that predicts field strength from what we
know about the transmitter — its power, height, frequency, and distance from the receiver.

This is called **regression** — predicting a continuous number, not a category.

---

*The data loading and longitude fix are the same patterns you used in Labs 11 and 12.*
*The new piece today is `scikit-learn`.*


---
## Setup

One new library today: **scikit-learn** — Python's standard machine learning toolkit.
You already have pandas, numpy, and folium from earlier labs.
Run this cell first, then wait for "Ready." before moving on.


In [ ]:
# Run this first
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "folium", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "scikit-learn", "-q"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

print("Ready.")


---
## Load the Data

1,510 FM stations measured from 25 receiver locations across the U.S., as depicted in RadioLand.
Each row is one transmitter–receiver pair with a measured field strength.


In [ ]:
URL = "https://raw.githubusercontent.com/CSC-2053-100-Fall25/python-ml-template/main/fm_stations_25_locations_ml_dataset.csv"
df = pd.read_csv(URL)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df[['callsign','city','search_location','distance','erp','haat','frequency','field_strength']].head(8)


### Fix Transmitter Longitude — Same as Labs 11 & 12

The transmitter `lon` column is stored as a positive number.
Negate it so coordinates are correct for the Western Hemisphere.
(The receiver `search_lon` is already negative — no fix needed there.)


In [ ]:
df['lon'] = df['lon'] * -1
print("Sample transmitter lons after fix:", df['lon'].head(3).tolist())


### Clean — Drop Rows with Missing Key Values

"Garbage in, garbage out." Models can't learn from rows where
the features or target are missing. We keep only rows where all
five key columns have valid, positive values.


In [ ]:
KEY = ['distance', 'erp', 'frequency', 'haat', 'field_strength']
df_clean = df.dropna(subset=KEY).copy()
df_clean = df_clean[(df_clean[KEY] > 0).all(axis=1)].reset_index(drop=True)
print(f"Rows after cleaning: {len(df_clean):,}  (removed {len(df) - len(df_clean)})")


---
## Explore — Know Your Target

**Rule #1 in ML: understand your data before you model it.**

The column we're predicting is `field_strength`, measured in **dBu**
(decibels above one microvolt per meter). The FCC uses this to measure
how strong a radio signal is at a specific receiving location.

- **Higher dBu** = stronger signal = clearer reception
- **Lower dBu** = weaker signal = static

A quick `.describe()` tells you the range, mean, and spread before you write a single model.
Fill in the blank with the column name.


In [ ]:
# Quick look at field_strength — given, just run it
print(df_clean['field_strength'].describe())


In [ ]:
# Distribution of field strength
plt.figure(figsize=(8, 4))
plt.hist(df_clean['field_strength'], bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Field Strength (dBu)')
plt.ylabel('Count')
plt.title('Distribution of Measured FM Signal Strength')
plt.tight_layout()
plt.show()


---
## Part 1 — Model 1: Distance Only

**The simplest possible question:** does signal strength drop as you get farther away?
Physics says yes — let's see what the data says.

---

### What is regression?

Regression is a type of machine learning where the goal is to predict a **continuous number**
— like signal strength, temperature, or house price — from a set of input features.

Under the hood, `LinearRegression` finds the best-fit line (or plane, or hyperplane)
through your training data. "Best-fit" means it minimizes the total squared error
between its predictions and the actual values. When you call `.fit()`, that's what's happening.

---

### Why do we split the data into train and test?

If you tested the model on the same data you trained it on, you'd be measuring
**memorization, not learning** — like grading a student on the exact questions
they studied. We hold out 20% of the data the model never sees during training,
then use that to measure how well it actually generalizes.

---

### The 5-step workflow — same every time:
1. Choose features `X` and target `y`
2. Split into train / test sets
3. **Fit** the model on training data (`.fit()`)
4. **Predict** on test data (`.predict()`)
5. **Evaluate** — how close were the predictions?


### Step 1 — Choose your features (X) and target (y)

**Features (X)** are the inputs the model is allowed to look at.
**Target (y)** is the number you want it to predict — in our case, `field_strength`.

Think of it like this: X is everything on the station's license application; y is the grade the FCC
would give its reception quality. The model's job is to learn the relationship between the two.

*Note on the double brackets:* `df[['distance']]` keeps X as a two-column-style table (2D).
`df['distance']` (single bracket) gives a flat list (1D). scikit-learn requires 2D for X — use double brackets.

In [ ]:
# Step 1 — Features and target
# X must be a DataFrame (2D), y must be a Series (1D)
X1 = df_clean[['___']]       # <-- fill in: just distance
y  = df_clean['field_strength']

print(f"X shape: {X1.shape}")
print(f"y shape: {y.shape}")


### Step 2 — Split into train and test sets

Before the model sees any data, we set aside 20% of it as a **test set** — data the model will
never train on. After training, we score it on that hidden 20% to see how well it generalizes
to stations it has never seen.

`random_state=42` just fixes the random shuffle so everyone in class gets the same split.
The four variables you name here will be used in every step that follows.

In [ ]:
# Step 2 — Train / test split (80% train, 20% test)
# Name your four variables: X1_train, X1_test, y1_train, y1_test
___, ___, ___, ___ = train_test_split(X1, y, test_size=0.2, random_state=42)

print(f"Training rows : {len(X1_train)}")
print(f"Test rows     : {len(X1_test)}")


### Step 3 — Fit the model

`LinearRegression()` creates a blank, untrained model.
`.fit(X_train, y_train)` is where the learning actually happens.

Internally, it finds the **best-fit line** through the training data — the line that minimizes
the total squared gap between its predictions and the real values. This is the same y = mx + b
you've seen before, but the model figures out m and b automatically from thousands of data points.

After fitting, `model.coef_[0]` is the slope — how much predicted field strength changes
for each additional kilometer of distance. You'd expect this to be **negative** (farther away = weaker signal).
`model.intercept_` is the y-intercept — the predicted field strength at zero distance.

In [ ]:
# Step 3 — Create and fit the model
model1 = LinearRegression()
model1.fit(___, ___)   # <-- fill in: X1_train, y1_train

coef = model1.coef_[0]
intercept = model1.intercept_
print(f"Learned equation: field_strength = {coef:.4f} × distance + {intercept:.2f}")


### Steps 4 & 5 — Predict, then evaluate

**Step 4:** `.predict()` runs the test features through the learned equation and returns
one predicted field strength for each station. The model has never seen these stations —
this is its first (and only) look at the test set.

**Step 5:** We compare those predictions to the real values using two scores:

- **R²** — ranges from 0 to 1. Think of it as "what percentage of the signal strength
  variation does the model explain?" R² = 0.48 means the model accounts for 48% of
  what's going on; the other 52% is noise or missing information.
- **RMSE** — the average error in the same units as the target (dBu). An RMSE of 16 dBu
  means the model's guesses are off by about 16 dBu on average.

In [ ]:
# Step 4 — Predict on the test set
y_pred1 = model1.predict(___)   # <-- fill in: X1_test

# Step 5 — Evaluate
r2_1   = r2_score(y1_test, y_pred1)
rmse_1 = np.sqrt(mean_squared_error(y1_test, y_pred1))

print(f"R²   : {r2_1:.3f}")
print(f"RMSE : {rmse_1:.1f} dBu")


In [ ]:
# Scatter: actual vs predicted
plt.figure(figsize=(6, 5))
plt.scatter(y1_test, y_pred1, alpha=0.4, s=15, color='steelblue')
plt.plot([y1_test.min(), y1_test.max()],
         [y1_test.min(), y1_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Field Strength (dBu)')
plt.ylabel('Predicted Field Strength (dBu)')
plt.title('Model 1: Distance Only')
plt.legend()
plt.tight_layout()
plt.show()


### What do R² and RMSE actually mean?

**R² (R-squared):** Measures how much of the variation in signal strength your model explains.
- R² = 1.0 → perfect predictions
- R² = 0.5 → model explains half the variation; the other half is noise or missing features
- R² = 0.0 → model is no better than always predicting the average

**RMSE (Root Mean Squared Error):** The average prediction error, in the same units as your target.
An RMSE of 16.4 dBu means the model's predictions are off by ~16 dBu on average.
Since the full range of field strength in this dataset is roughly 100 dBu (40–140),
that's a meaningful error — there's clearly something distance alone isn't capturing.

---

**Q1 — Interpreting Model 1**

R² ≈ 0.48 and RMSE ≈ 16.4 dBu. The model's distance coefficient is negative —
does that make physical sense? What else might explain the other 52% of the variation?

*(Double-click to answer.)*

> **Your answer:**


---
## Part 2 — Model 2: Add Power, Frequency, and Height

Distance alone explained ~48% of the variation. What's in the other 52%?

A 100 kW transmitter obviously reaches farther than a 1 kW one.
An antenna mounted on a mountain clears terrain that a rooftop antenna can't.
Higher frequencies behave differently than lower ones.

Let's add three more features the FCC measures for every licensed station:

| Feature | Column | What it means |
|---|---|---|
| Effective Radiated Power | `erp` | How much power the transmitter puts out (kW) |
| FM Frequency | `frequency` | The dial position in MHz (e.g., 93.5) |
| Height Above Average Terrain | `haat` | How high the antenna is above surrounding terrain (meters) |

Adding features to a linear model doesn't require any structural change —
just pass a wider `X` matrix. The model finds the best coefficient for each one.


In [ ]:
# Step 1 — Four features this time
FEATURES2 = ['___', '___', '___', '___']   # distance, erp, frequency, haat

X2 = df_clean[FEATURES2]

# Steps 2–4 — split, fit, predict
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X2_train, y2_train)
y_pred2 = model2.predict(X2_test)

r2_2   = r2_score(y2_test, y_pred2)
rmse_2 = np.sqrt(mean_squared_error(y2_test, y_pred2))
print(f"R²   : {r2_2:.3f}")
print(f"RMSE : {rmse_2:.1f} dBu")


In [ ]:
# What did the model learn? — feature coefficients
coef_df = pd.DataFrame({
    'feature':     FEATURES2,
    'coefficient': model2.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df.to_string(index=False))


### Reading the coefficient table

Each coefficient tells you: *"for every one-unit increase in this feature,
how much does predicted field strength change, holding everything else constant?"*

- **Negative coefficient** → that feature weakens the signal as it increases
- **Positive coefficient** → that feature strengthens the signal as it increases
- **Larger absolute value** → that feature has more influence on the prediction

**Q2 — Which feature matters most?**

Do all four signs make physical sense? `distance` should be negative (farther = weaker).
What do the signs of `erp`, `haat`, and `frequency` tell you?
Which single feature has the largest effect on the prediction?

*(Double-click to answer.)*

> **Your answer:**


---
## Part 3 — Feature Engineering: Giving the Model Better Inputs

R² jumped from 0.48 → 0.74 by adding three features. Can we do better
without adding any new data columns?

**Feature engineering** means creating new columns derived from the ones you already have —
transforming the data into a shape the model can use more effectively.

---

### The Physics: Inverse Square Law

Radio signal power doesn't decrease linearly with distance — it decreases with the
**square** of distance. Double the distance, and the signal drops to **one quarter** strength:

```
Power ∝ 1 / distance²
```

This is a curved (non-linear) relationship. `LinearRegression` fits straight lines.
If the true relationship is curved, the model will never fit it well using raw distance.

**The fix:** take the log. `log(distance²)` = `2 × log(distance)`.
This converts the curve into a line — which a linear model can fit perfectly.

We do the same for ERP (power), which also has a non-linear relationship with field strength.
We also add `distance²` directly, giving the model the squared term explicitly.

Fill in the two `np.log()` formulas below. Add 1 inside the log to avoid `log(0)`.


In [ ]:
df_eng = df_clean.copy()

# Fill in the formulas
df_eng['log_distance'] = np.log(___ + 1)   # log transform of distance
df_eng['log_erp']      = np.log(___ + 1)   # log transform of erp
df_eng['distance_sq']  = df_eng['distance'] ** 2   # given — inverse square law

print("Sample engineered features:")
print(df_eng[['distance','log_distance','erp','log_erp','distance_sq']].head(4).to_string(index=False))


In [ ]:
# Train Model 3 with original + engineered features
FEATURES3 = ['distance', 'erp', 'frequency', 'haat',
             'log_distance', 'log_erp', 'distance_sq']

X3 = df_eng[FEATURES3]
y3 = df_eng['field_strength']

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=42)

model3 = LinearRegression()
model3.fit(X3_train, y3_train)
y_pred3 = model3.predict(X3_test)

r2_3   = r2_score(y3_test, y_pred3)
rmse_3 = np.sqrt(mean_squared_error(y3_test, y_pred3))
print(f"R²   : {r2_3:.3f}")
print(f"RMSE : {rmse_3:.1f} dBu")


In [ ]:
# Compare all three models — given, just run it
results = pd.DataFrame({
    'Model':    ['1: Distance only', '2: + Power/Freq/Height', '3: + Log transforms'],
    'Features': [1, 4, 7],
    'R²':       [r2_1, r2_2, r2_3],
    'RMSE (dBu)': [rmse_1, rmse_2, rmse_3],
})
print(results.to_string(index=False))

plt.figure(figsize=(7, 4))
bars = plt.bar(results['Model'], results['R²'], color=['#aec6cf','#5b92b5','#1b3a6b'])
plt.ylabel('R² Score')
plt.title('Model Comparison — R² by Feature Set')
plt.ylim(0, 1)
for bar, val in zip(bars, results['R²']):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.2f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.show()


**Q3 — The improvement arc**

| Model | R² | RMSE | What changed |
|---|---|---|---|
| 1: Distance only | 0.48 | 16.4 dBu | Baseline |
| 2: + Power/Freq/Height | 0.74 | 11.6 dBu | More real-world features |
| 3: + Log transforms | 0.89 | 7.5 dBu | Better math, same data |

The FCC considers a field strength of ~60 dBu a "protected" service contour for FM.
A 7.5 dBu average error is meaningful — you could mistake a usable signal for a fringe one.
But compared to where we started (16.4 dBu), it's a big improvement from just transforming
two columns.

Why did the log transforms help so much more than simply adding more raw features?
What does that tell you about the relationship between distance and signal strength?

*(Double-click to answer.)*

> **Your answer:**


---
## Part 4 — Map Your Predictions (Compare With a Partner)

Use Model 3 to predict signal strength for every station receivable from **New York, NY**.
Then map them — color each transmitter by how close the model's prediction was.

- **Green** → within 5 dBu (good prediction)
- **Orange** → model over-predicted by more than 5 dBu (too optimistic)
- **Red** → model under-predicted by more than 5 dBu (too pessimistic)


In [ ]:
# Filter to New York receiver
ny = df_eng[df_eng['search_location'] == 'New York, NY'].copy()
print(f"Stations receivable from New York: {len(ny)}")

# Predict with Model 3
ny_pred = model3.predict(ny[FEATURES3])
ny['predicted'] = ny_pred
ny['error'] = ny['predicted'] - ny['field_strength']   # positive = over-predicted

print(f"Mean error: {ny['error'].mean():.1f} dBu")
print(f"RMSE on NY: {np.sqrt((ny['error']**2).mean()):.1f} dBu")


In [ ]:
# Build the map — fill in the color logic
m = folium.Map(location=[40.71, -74.01], zoom_start=8)

for _, row in ny.iterrows():
    # Fill in the blank: assign color based on error
    if abs(row['error']) <= 5:
        color = '___'           # good prediction  → 'green'
    elif row['error'] > 5:
        color = '___'           # over-predicted   → 'orange'
    else:
        color = '___'           # under-predicted  → 'red'

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.75,
        tooltip=(
            f"{row['callsign']} ({row['frequency']} MHz)<br>"
            f"Actual: {row['field_strength']:.1f} dBu<br>"
            f"Predicted: {row['predicted']:.1f} dBu<br>"
            f"Error: {row['error']:+.1f} dBu"
        )
    ).add_to(m)

display(m)


**Q4 — Where does the model go wrong?**

Look at the map. Are the red/green stations clustered somewhere geographically,
or scattered randomly? What might explain systematic over- or under-prediction
in certain areas? (Think: terrain, urban canyons, interference.)

*(Double-click to answer.)*

> **Your answer:**


---
## Q5 — So What? The Engineering Question

You've built a model that predicts FM signal strength with an average error of ~7.5 dBu
using nothing but publicly available FCC license data.

Think about that from a broadcast engineer's perspective:

- **Coverage planning:** A station applying for a license or a power increase needs to
  demonstrate it won't interfere with existing stations. Could a model like this help
  estimate coverage contours before filing with the FCC?
- **Interference analysis:** If a new station is proposed in your market, could you use
  a model like this to predict how much it would affect your signal at key locations?
- **Missing features:** What does a real broadcast engineer know about a signal that
  this model doesn't? (Think: terrain between transmitter and receiver, antenna pattern,
  building penetration, receiver quality.)

**In 3–5 sentences:** If you handed this model to a radio station engineer, what is one
specific decision it could help them make — and what would it get wrong?

*(Double-click to answer.)*

> **Your answer:**


---
## Stretch — Try Another City

Copy the map code above into the cell below and change the city to one of these:

- `'Atlanta, GA'`
- `'Los Angeles, CA'`
- `'Anchorage, AK'`
- `'Key West, FL'`

Does the model perform better or worse? Does the error pattern look different?


In [ ]:
# Stretch — your city here
CITY = '___'

city_df = df_eng[df_eng['search_location'] == CITY].copy()
city_pred = model3.predict(city_df[FEATURES3])
city_df['predicted'] = city_pred
city_df['error'] = city_df['predicted'] - city_df['field_strength']

print(f"{CITY}: {len(city_df)} stations")
print(f"RMSE: {np.sqrt((city_df['error']**2).mean()):.1f} dBu")

# YOUR MAP CODE HERE
